# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


In [3]:
%idle_timeout 2880
%glue_version 3.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)


You are already connected to a glueetl session 01db93f6-be84-49ad-9ce3-12cdef81ac6a.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session 01db93f6-be84-49ad-9ce3-12cdef81ac6a.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 3.0


You are already connected to a glueetl session 01db93f6-be84-49ad-9ce3-12cdef81ac6a.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: None
Setting new worker type to: G.1X


You are already connected to a glueetl session 01db93f6-be84-49ad-9ce3-12cdef81ac6a.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: None
Setting new number of workers to: 5



In [1]:
spark.sql("SHOW DATABASES").show()

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 01db93f6-be84-49ad-9ce3-12cdef81ac6a
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 01db93f6-be84-49ad-9ce3-12cdef81ac6a to get into ready status...
Session 01db93f6-be84-49ad-9ce3-12cdef81ac6a has been created.
+----------------+
|       namespace|
+----------------+
|         default|
|`imba-tobby-17a`|
|         imba_db|
+----------------+


In [4]:
# ── Cell 2: Load raw tables from Glue Catalog ──────────────────────────────────
dy_orders = glueContext.create_dynamic_frame.from_catalog(database='imba-tobby-17a', table_name='orders')
df_orders = dy_orders.toDF()
df_orders.createOrReplaceTempView('orders')
df_orders.show()

dy_order_products = glueContext.create_dynamic_frame.from_catalog(database='imba-tobby-17a', table_name='order_products')
df_order_products = dy_order_products.toDF()
df_order_products.createOrReplaceTempView('order_products')
df_order_products.show()

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  null|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
| 3367565|      1|   prior|           6|        2|                7|                  19.0|
|  550135|      1|   prior|           7|        1|                9|                  20.0|
| 3108588|      1|   prior|           8|        1|               14|            

In [5]:
# ── Cell 3: Create order_products_prior ────────────────────────────────────────
df_opp = spark.sql("""
    SELECT a.*,
           b.product_id,
           b.add_to_cart_order,
           b.reordered
    FROM orders a
    JOIN order_products b ON a.order_id = b.order_id
    WHERE a.eval_set = 'prior'
""")
df_opp.createOrReplaceTempView('order_products_prior')
df_opp.show()

df_opp.write.mode('overwrite').format('parquet').save('s3://imba-tobby-17a/features/order_products_prior_sv/')


+--------+-------+--------+------------+---------+-----------------+----------------------+----------+-----------------+---------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_id|add_to_cart_order|reordered|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+-----------------+---------+
|      29|  81606|   prior|          14|        5|               12|                  10.0|     43352|                1|        1|
|      93| 130239|   prior|          15|        6|               10|                  19.0|     27323|                8|        1|
|     208|  68303|   prior|           7|        4|               13|                   6.0|     34503|                4|        1|
|     221| 177947|   prior|           1|        5|                8|                  null|     24010|                1|        0|
|     224| 109534|   prior|           6|        0|               14|               

In [6]:
# ── Cell 4: user_features_1 ───────────────────────────────────────────────────
df_uf1 = spark.sql("""
    SELECT user_id,
           Max(order_number)           AS user_orders,
           Sum(days_since_prior_order) AS user_period,
           Avg(days_since_prior_order) AS user_mean_days_since_prior
    FROM orders
    GROUP BY user_id
""")
df_uf1.show()

df_uf1.write.mode('overwrite').format('parquet').save('s3://imba-tobby-17a/features/user_features_1_sv/')


+-------+-----------+-----------+--------------------------+
|user_id|user_orders|user_period|user_mean_days_since_prior|
+-------+-----------+-----------+--------------------------+
| 128024|         26|      315.0|                      12.6|
| 128071|          7|      171.0|                      28.5|
| 128083|          7|      109.0|        18.166666666666668|
| 128085|         47|      363.0|         7.891304347826087|
| 128087|         15|      289.0|        20.642857142857142|
| 128121|          7|      159.0|                      26.5|
| 128127|          6|      126.0|                      25.2|
| 128135|          4|       50.0|        16.666666666666668|
| 128203|         12|      122.0|        11.090909090909092|
| 128218|         46|      360.0|                       8.0|
| 128223|         34|      196.0|        5.9393939393939394|
| 128232|          8|      146.0|        20.857142857142858|
| 128237|         18|      310.0|        18.235294117647058|
| 128295|          9|   

In [7]:
# ── Cell 5: user_features_2 ───────────────────────────────────────────────────
df_uf2 = spark.sql("""
    SELECT user_id,
           Count(*)                   AS user_total_products,
           Count(DISTINCT product_id) AS user_distinct_products,
           Sum(CASE WHEN reordered = 1 THEN 1 ELSE 0 END)
             / Cast(Sum(CASE WHEN order_number > 1 THEN 1 ELSE 0 END) AS DOUBLE)
                                      AS user_reorder_ratio
    FROM order_products_prior
    GROUP BY user_id
""")
df_uf2.show()

df_uf2.write.mode('overwrite').format('parquet').save('s3://imba-tobby-17a/features/user_features_2_sv/')


+-------+-------------------+----------------------+-------------------+
|user_id|user_total_products|user_distinct_products| user_reorder_ratio|
+-------+-------------------+----------------------+-------------------+
| 102119|                331|                   179|0.48562300319488816|
|  40821|                604|                   163| 0.7577319587628866|
| 188743|                228|                   123| 0.4666666666666667|
|  81404|                260|                   158| 0.4031620553359684|
| 133215|                227|                    50| 0.8082191780821918|
|  56387|                617|                   194| 0.7038269550748752|
|  90183|                 17|                    10| 0.4666666666666667|
| 138753|                272|                    85| 0.7276264591439688|
| 175351|                131|                    77|0.43548387096774194|
| 189823|                509|                   139| 0.7341269841269841|
| 163853|                 51|                    20

In [8]:
# ── Cell 6: up_features ───────────────────────────────────────────────────────
df_up = spark.sql("""
    SELECT user_id,
           product_id,
           Count(*)               AS up_orders,
           Min(order_number)      AS up_first_order,
           Max(order_number)      AS up_last_order,
           Avg(add_to_cart_order) AS up_average_cart_position
    FROM order_products_prior
    GROUP BY user_id, product_id
""")
df_up.show()

df_up.write.mode('overwrite').format('parquet').save('s3://imba-tobby-17a/features/up_features_sv/')


+-------+----------+---------+--------------+-------------+------------------------+
|user_id|product_id|up_orders|up_first_order|up_last_order|up_average_cart_position|
+-------+----------+---------+--------------+-------------+------------------------+
| 117132|     27344|        1|            18|           18|                     1.0|
| 172032|      2120|        4|             8|           19|                    12.5|
| 195701|     27845|        2|             1|            2|                     4.5|
|  33495|     21903|       31|             3|           95|        9.67741935483871|
| 198103|     19382|       32|             5|           65|                  5.5625|
| 124653|     46359|        1|             3|            3|                     3.0|
| 163644|      4289|       18|             1|           59|       4.833333333333333|
| 160502|     26209|        3|             6|           30|       6.666666666666667|
| 129992|     16059|        4|             1|            5|      

In [9]:
# ── Cell 7: prd_features ──────────────────────────────────────────────────────
df_prd = spark.sql("""
    SELECT product_id,
           Count(*)                                               AS prod_orders,
           Sum(reordered)                                         AS prod_reorders,
           Sum(CASE WHEN product_seq_time = 1 THEN 1 ELSE 0 END) AS prod_first_orders,
           Sum(CASE WHEN product_seq_time = 2 THEN 1 ELSE 0 END) AS prod_second_orders
    FROM (
        SELECT *,
               Rank() OVER (PARTITION BY user_id, product_id ORDER BY order_number) AS product_seq_time
        FROM order_products_prior
    )
    GROUP BY product_id
""")
df_prd.show()

df_prd.write.mode('overwrite').format('parquet').save('s3://imba-tobby-17a/features/prd_features_sv/')


+----------+-----------+-------------+-----------------+------------------+
|product_id|prod_orders|prod_reorders|prod_first_orders|prod_second_orders|
+----------+-----------+-------------+-----------------+------------------+
|       248|       6371|         2550|             3821|              1068|
|     18370|      18449|        11140|             7309|              3235|
|     10500|       1254|          433|              821|               184|
|      6187|      15415|         9389|             6026|              2697|
|     33731|      45238|        27635|            17603|              8226|
|     48739|        291|           79|              212|                55|
|     38741|         69|           32|               37|                12|
|     33002|        255|          146|              109|                44|
|     35833|        770|          296|              474|               128|
|     10831|       8337|         6677|             1660|              1019|
|      3793|

#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)